# 📘 Slowly Changing Dimensions (SCD) Type 1
**🔹 What is SCD?**

**Slowly Changing Dimensions (SCD) in a data warehouse are techniques used to manage and store how dimensional data (like customer addresses or product prices) changes over time.** They ensure data integrity and historical tracking for accurate reporting. Key types include Type 1 (overwriting), Type 2 (adding new rows), and Type 3 (using current/previous columns).

## 1. 🔹 What is SCD Type 1?

SCD Type 1 is the simplest approach:

👉 It overwrites old data with new data — no history is preserved.

🔹 Key Idea
- Only current/latest value is stored
- No tracking of historical changes
- Updates happen in-place

🔹 Example
🧾 Customer Table (Before Update)

| customer_id | name | city    |
| ----------- | ---- | ------- |
| 101         | Ravi | Kolkata |


🔄 Change Event

Customer moves from Kolkata → Mumbai

🧾 After SCD Type 1 Update

| customer_id | name | city   |
| ----------- | ---- | ------ |
| 101         | Ravi | Mumbai |

👉 Old value ("Kolkata") is lost

🔹 How It Works (Conceptual Flow)
1. Identify incoming record
2. Match with existing record (using primary key)
3. If match found:
       UPDATE existing record
4. If no match:
       INSERT new record

🔹 SQL Example

```sql
MERGE INTO customer_dim AS target
USING customer_stage AS source
ON target.customer_id = source.customer_id

WHEN MATCHED THEN
  UPDATE SET
    target.name = source.name,
    target.city = source.city

WHEN NOT MATCHED THEN
  INSERT (customer_id, name, city)
  VALUES (source.customer_id, source.name, source.city);
```


🔹 When to Use SCD Type 1

Use it when:

- Historical data is not important
- Only latest value matters
- Corrections are needed (e.g., fixing wrong data)

📌 Examples:
- Fixing spelling mistakes in names
- Updating incorrect email IDs
- Correcting product descriptions


🔹 When NOT to Use

Avoid SCD Type 1 if:

- You need audit/history tracking
- Regulatory compliance requires history
- Business needs trend analysis

🔹 Comparison with Other SCD Types
| Type   | Behavior                         |
| ------ | -------------------------------- |
| Type 1 | Overwrite data                   |
| Type 2 | Add new row (full history)       |
| Type 3 | Add new column (limited history) |


In [0]:
%sql
CREATE DATABASE hive_metastore.hive_db;

CREATE TABLE IF NOT EXISTS hive_metastore.hive_db.hive_table;

In [0]:
from delta.tables import DeltaTable

data=[(1000,'Rakesh','Sharma','India'),
       (2000,'Arijit','Jana','Pakistan'),
       (3000,'Ravi','Kumar','Iran')]

df = spark.createDataFrame(data,['personId','firstname','lastname','country'])

# Creating DeltaTable Object using .forPath(), we can also use .forName()

In [0]:
df.write.format('delta').mode('overwrite').partitionBy("personId").option("overwriteSchema", "true").saveAsTable("hive_metastore.hive_db.persons")
# delta_table_path="dbfs:/user/hive/warehouse/demo.db/persons"
delta_table_path = "dbfs:/user/hive/warehouse/hive_db.db/persons"
deltaTableOriginal = DeltaTable.forPath(spark, delta_table_path)

In [0]:
df1 = spark.read.table('hive_metastore.hive_db.persons')
df1.show()

In [0]:
# Create a List Containing the "Column Names" for "Person"
personColumns = ['personId','firstname','lastname','country']

# Create a List Containing the "Data" for "Person"
personList = [\
                
                (1006,"Soham","Sarkar","ENG"),
                (1009,"Pinaki","Ganguly","IND"),
                (1007,"Rajesh","Gupta","USA"),
                (1008,"Rajesh","Kumar","USA"),
                (2000,'Arijit','Jana','India'),
                (3000,'Ravi','Kumar','USA')

             ]
dfPerson = spark.createDataFrame(personList, schema = personColumns)
# dfPerson.createOrReplaceTempView("vw_person")

In [0]:
dfPerson.show()

# Using whenMatchedUpdateAll() and whenNotMatchedInserAll() function

In [0]:
# deltaTableOriginal.alias('t').merge(dfPerson.alias('s'),'t.personId = s.personId')\
#     .whenMatchedUpdateAll()\
#         .whenNotMatchedInsertAll()\
#             .execute()

# Using whenMatchedUpdate(set = {}) and whenNotMatchedInsert(values = {}) function

In [0]:
# Before Upsert DF Result
# -----------------------
# -----------------------

# df_before = deltaTableOriginal.toDF()
# df_before.show()

# ------------ OR ---------------- #

df_before = spark.read.table('hive_metastore.hive_db.persons')
df_before.show()

In [0]:
deltaTableOriginal.alias('t').merge(dfPerson.alias('s'),'t.personId = s.personId')\
    .whenMatchedUpdate(
        set = {
            "personId": "s.personId",
            "firstname": "s.firstname",
            "lastname": "s.lastname",
            "country": "s.country"
        }
    )\
    .whenNotMatchedInsert(
        values = {
            "personId": "s.personId",
            "firstname": "s.firstname",
            "lastname": "s.lastname",
            "country": "s.country"
        }
    )\
    .execute()

In [0]:
# After Upsert DF Result
# -----------------------
# -----------------------

df_after = deltaTableOriginal.toDF()
df_after.show()

# ------------ OR ---------------- #

# df_after = spark.read.table('hive_metastore.hive_db.persons')
# df_after.show()

In [0]:
spark

# Pinaki's Code

In [0]:
# deltaTableOriginal.alias('people') \
#   .merge(
#     dfPerson.alias('updates'),
#     'people.PersonId = updates.PersonId'
#   ) \
#   .whenMatchedUpdate(set =
#     {
#       "Personid": "updates.PersonId",
#       "firstName": "updates.FirstName",
#       "lastName": "updates.LastName",
#     }
#   ) \
#   .whenNotMatchedInsert(values =
#     {
#       "Personid": "updates.PersonId",
#       "firstName": "updates.FirstName",
#       "lastName": "updates.LastName",
#       "Country": "updates.Country"
#     }
#   ) \
#   .execute()